# Notebook 05 — Analyzing BindCraft Output with Pandas

**Learning objectives:**
- Load and explore BindCraft CSV files using pandas
- Sort, filter, and compute statistics on design metrics
- Plot metric distributions
- Apply the traffic light system to select top designs

**Time:** ~60 minutes

**Companion wiki pages:** [7.1 Metrics](https://rucha1796.github.io/internship-bindcraft-2026/m7_01_metrics/) | [7.2 Ranking](https://rucha1796.github.io/internship-bindcraft-2026/m7_02_ranking/)

---
> No GPU required. Works with the pregenerated PDL1_8aok dataset.

## Cell 1 — Load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Mount Drive if not already mounted
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

# ============================================================
# DATA SOURCE — point to your BindCraft output folder
DATA_DIR = "/content/drive/MyDrive/bindcraft/PDL1_8aok"   # pregenerated
# DATA_DIR = "/content/drive/MyDrive/bindcraft/PDL1"       # your own run
# ============================================================

csv_path = f"{DATA_DIR}/final_design_stats.csv"

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"✓ Loaded {len(df)} designs from {csv_path}")
    print(f"  Columns: {len(df.columns)}")
else:
    # Fallback demo data
    print("Dataset not found — generating demo data")
    np.random.seed(42)
    n = 12
    df = pd.DataFrame({
        "binder_name": [f"PDL1_traj{i}_mpnn1" for i in range(1, n+1)],
        "binder_sequence": ["MKVIFGLMVGGVIAFVLG" + "AKE"*i for i in range(1, n+1)],
        "Average_i_pTM": np.random.uniform(0.56, 0.74, n),
        "Average_pLDDT": np.random.uniform(0.86, 0.94, n),
        "Average_Binder_pLDDT": np.random.uniform(0.81, 0.91, n),
        "Average_Binder_RMSD": np.random.uniform(0.5, 1.9, n),
        "Average_ShapeComplementarity": np.random.uniform(0.56, 0.72, n),
        "Average_dG": np.random.uniform(-24, -11, n),
        "Average_i_pAE": np.random.uniform(5, 9.5, n),
        "Average_n_InterfaceHbonds": np.random.randint(3, 13, n).astype(float),
        "Average_Relaxed_Clashes": np.random.uniform(0, 4.5, n),
    })
    print(f"✓ Demo dataset created with {n} designs")

## Cell 2 — Quick overview

In [ ]:
KEY_METRICS = [
    "Average_i_pTM",
    "Average_pLDDT",
    "Average_Binder_pLDDT",
    "Average_Binder_RMSD",
    "Average_ShapeComplementarity",
    "Average_dG",
    "Average_i_pAE",
    "Average_n_InterfaceHbonds",
    "Average_Relaxed_Clashes"
]

# Keep only columns that exist in the dataset
available_metrics = [c for c in KEY_METRICS if c in df.columns]

print("=== Dataset Overview ===")
print(f"Total designs:  {len(df)}")
print(f"\nStatistics for key metrics:")
print(df[available_metrics].describe().round(3).T.to_string())

## Cell 3 — Ranked table

In [ ]:
df_ranked = df.sort_values("Average_i_pTM", ascending=False).reset_index(drop=True)
df_ranked.index += 1  # Start ranking from 1

display_cols = ["binder_name"] + available_metrics

pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.max_colwidth", 30)

print("All designs ranked by i_pTM (highest first):")
print(df_ranked[display_cols].to_string())

## Cell 4 — Traffic light system

Apply the thresholds from Module 7.1 to score each design. Green = pass, Red = borderline.

In [ ]:
THRESHOLDS = {
    "Average_i_pTM":                 ("≥", 0.55),
    "Average_pLDDT":                  ("≥", 0.85),
    "Average_Binder_pLDDT":          ("≥", 0.80),
    "Average_i_pAE":                  ("≤", 10.0),
    "Average_Binder_RMSD":            ("≤", 2.0),
    "Average_ShapeComplementarity":   ("≥", 0.55),
    "Average_dG":                     ("≤", -10.0),
    "Average_Relaxed_Clashes":        ("≤", 5.0),
}

STRONG_THRESHOLDS = {
    "Average_i_pTM":                 ("≥", 0.65),
    "Average_ShapeComplementarity":   ("≥", 0.65),
    "Average_Binder_RMSD":            ("≤", 1.0),
    "Average_dG":                     ("≤", -20.0),
}

def traffic_light(row, thresholds):
    """Return number of metrics passing threshold for a row."""
    passing = 0
    for col, (op, threshold) in thresholds.items():
        if col not in row:
            continue
        val = row[col]
        if op == "≥" and val >= threshold:
            passing += 1
        elif op == "≤" and val <= threshold:
            passing += 1
    return passing

df_ranked["green_count"] = df_ranked.apply(lambda r: traffic_light(r, THRESHOLDS), axis=1)
df_ranked["strong_count"] = df_ranked.apply(lambda r: traffic_light(r, STRONG_THRESHOLDS), axis=1)

n_threshold = len([k for k in THRESHOLDS if k in df_ranked.columns])
n_strong = len([k for k in STRONG_THRESHOLDS if k in df_ranked.columns])

print(f"Traffic light system ({n_threshold} criteria):")
print(f"  Green count = how many metrics PASS the standard threshold")
print(f"  Strong count = how many metrics PASS the higher (excellent) threshold\n")

summary_cols = ["binder_name", "Average_i_pTM", "Average_ShapeComplementarity",
                "Average_Binder_RMSD", "green_count", "strong_count"]
print(df_ranked[summary_cols].to_string())

## Cell 5 — Metric distribution plots

In [ ]:
plot_metrics = [
    ("Average_i_pTM", 0.55, "≥"),
    ("Average_ShapeComplementarity", 0.55, "≥"),
    ("Average_Binder_RMSD", 2.0, "≤"),
    ("Average_dG", -10.0, "≤"),
    ("Average_Binder_pLDDT", 0.80, "≥"),
    ("Average_n_InterfaceHbonds", 5, "≥"),
]
plot_metrics = [(m, t, op) for m, t, op in plot_metrics if m in df.columns]

n_cols = 3
n_rows = (len(plot_metrics) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for ax, (col, threshold, op) in zip(axes, plot_metrics):
    vals = df[col].dropna()
    ax.hist(vals, bins=min(15, len(vals)), color="#3c5b6f", edgecolor="white", alpha=0.85)
    ax.axvline(threshold, color="red", linestyle="--", linewidth=2,
               label=f"threshold ({op}{threshold})")
    ax.axvline(vals.mean(), color="orange", linestyle=":", linewidth=1.5,
               label=f"mean ({vals.mean():.3f})")
    ax.set_title(col.replace("Average_", ""), fontsize=11)
    ax.legend(fontsize=8)

# Hide unused axes
for ax in axes[len(plot_metrics):]:
    ax.set_visible(False)

plt.suptitle(f"BindCraft Metric Distributions ({len(df)} accepted designs)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("metric_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: metric_distributions.png")

## Cell 6 — Scatter plot: i_pTM vs. ShapeComplementarity

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

if "Average_Binder_RMSD" in df.columns:
    scatter = ax.scatter(
        df["Average_i_pTM"],
        df["Average_ShapeComplementarity"],
        c=df["Average_Binder_RMSD"],
        cmap="RdYlGn_r",
        s=120, edgecolors="black", linewidth=0.5,
        vmin=0.5, vmax=2.0
    )
    plt.colorbar(scatter, ax=ax, label="Binder RMSD (Å) — green=good")
else:
    ax.scatter(df["Average_i_pTM"], df["Average_ShapeComplementarity"],
               s=120, color="#3c5b6f", edgecolors="black")

ax.axvline(0.55, color="gray", linestyle="--", alpha=0.7, label="i_pTM threshold")
ax.axhline(0.55, color="gray", linestyle=":", alpha=0.7, label="ShapeComp threshold")

# Label top 5 by i_pTM
top5_idx = df["Average_i_pTM"].nlargest(5).index
for idx in top5_idx:
    row = df.loc[idx]
    ax.annotate(
        str(df["Average_i_pTM"].rank(ascending=False).astype(int).loc[idx]),
        xy=(row["Average_i_pTM"], row["Average_ShapeComplementarity"]),
        xytext=(5, 5), textcoords="offset points", fontsize=9, fontweight="bold"
    )

ax.set_xlabel("Average i_pTM", fontsize=12)
ax.set_ylabel("Average Shape Complementarity", fontsize=12)
ax.set_title("Design quality: i_pTM vs. Shape Complementarity\n(color = Binder RMSD; numbers = i_pTM rank)", fontsize=11)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("quality_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print("Best designs are in the top-right quadrant (high i_pTM + high ShapeComp) with green color (low RMSD)")

## Cell 7 — Select your top 5

In [ ]:
# Sort by i_pTM descending, then ShapeComplementarity descending
sort_cols = [c for c in ["Average_i_pTM", "Average_ShapeComplementarity"] if c in df.columns]
top5 = df.sort_values(sort_cols, ascending=False).head(5).reset_index(drop=True)
top5.index += 1

print("=" * 80)
print("TOP 5 RECOMMENDED DESIGNS FOR SYNTHESIS")
print("=" * 80)
print()

for rank, row in top5.iterrows():
    print(f"Rank {rank}: {row['binder_name']}")
    print(f"  i_pTM:               {row.get('Average_i_pTM', 'N/A'):.3f}")
    print(f"  Shape Complementarity: {row.get('Average_ShapeComplementarity', 'N/A'):.3f}")
    print(f"  Binder RMSD:          {row.get('Average_Binder_RMSD', 'N/A'):.2f} Å")
    print(f"  dG:                   {row.get('Average_dG', 'N/A'):.1f} kcal/mol")
    if "binder_sequence" in row:
        seq = row["binder_sequence"]
        print(f"  Sequence ({len(seq)} aa):   {seq[:40]}...")
    print()

## Cell 8 — Sequence diversity check

Make sure you're not recommending 5 nearly-identical sequences.

In [ ]:
if "binder_sequence" in top5.columns:
    from difflib import SequenceMatcher
    
    seqs = top5["binder_sequence"].tolist()
    names = top5["binder_name"].tolist()
    
    print("Pairwise sequence similarity (top 5 designs):")
    print("(> 90% similar = nearly identical; prefer diverse selections)\n")
    
    for i in range(len(seqs)):
        for j in range(i+1, len(seqs)):
            sim = SequenceMatcher(None, seqs[i], seqs[j]).ratio()
            flag = " ⚠ very similar!" if sim > 0.9 else ""
            print(f"  Design {i+1} vs Design {j+1}: {sim*100:.0f}% similar{flag}")
else:
    print("binder_sequence column not available in this dataset.")
    print("Sequence diversity check requires the full CSV with sequences.")

## Cell 9 — Reflection questions

Fill in your answers. These will form the basis of your Design Recommendation Report.

**Q1:** What is the range of i_pTM across all accepted designs? Are they clustered near the threshold or well above it?

_Your answer:_

**Q2:** Which metric has the widest spread across designs? What does this tell you?

_Your answer:_

**Q3:** In the scatter plot, which quadrant has most designs? Are there any outliers?

_Your answer:_

**Q4:** For your Rank 1 design, what makes it the best? Name at least two metrics where it stands out.

_Your answer:_

**Q5:** Are your top 5 sequences diverse? If two are very similar, which would you drop and replace with a more diverse pick, and why?

_Your answer:_

---
**Next:** Notebook 06 — Visualizing your designed binders in 3D